# Downsampling data

The number of images best explained with every interpretation technique is as follows:
- IG: 147 (74% of total images)
- XRAI: 32 (16% of total images)
- LIME: 18 (9% of total images)
- ANCHOR: 1 (>1% of total images)
- Total images: 198

Due to a majority of IG images, prediction for new images is likely to be biased towards the IG technique. Is is neccesary to smoth the proportions of elected techniques in the dataset.
In this notebook, we adress that problem by downsampling our dataset (i.e. picking a subset of instances that smoths the proportion of techniques).

We'll substract IG images from the dataset, so the proportion of the IG technique in the dataset is of ~60% instead of 74%, allowing other techniques to be more present during the prediction of interpretation techniques for new images. The number of images after downsampling should look like this:

- IG: 77 (60% of total images, 52% of original IG count)
- XRAI: 32 (25% of total images)
- LIME: 18 (14% of total images)
- ANCHOR: 1 (>1% of total images)
- Total images: 128

If we try to lower IG proportions to ~50%, the downsampled dataset would look like this:
- IG: 51 (50% of total images, 35% of original IG count)
- XRAI: 32 (31% of total images)
- LIME: 18 (18% of total images)
- ANCHOR: 1 (1% of total images)
- Total images: 102

We'll only use the latent features dataset now, because other similarity metrics do not perform as well as euclidian latent feature distance. However, is possible to extend this notebook to perform subsampling on datasets based on other similarity metrics.

### Some subsampling constants

In [ ]:
p = 0.55 # DESIRED_IG_PROP
ig_rem = (147-198*p)/(1-p) # No. of IG imgs. to remove
ig_keep = 147 - ig_rem
ig_pct_keep = 1 - ig_rem/147 # % of IG imgs. to keep
new_len = 198 - ig_rem # New lenght for downsampled dataset
print(ig_rem, ig_keep, ig_pct_keep, new_len)

## Loading Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sklearn

In [ ]:
FEATS_FILE_PATH = os.path.join('..', 'features', 'incv1_feats.csv')
BEST_TECHNIQUES = os.path.join('..', 'results', 'votes_summary.csv')

In [ ]:
feats_df = pd.read_csv(FEATS_FILE_PATH, index_col=0)
feats_df.head(3)

In [ ]:
techniques_df = pd.read_csv(BEST_TECHNIQUES, dtype='object', index_col=0)
techniques_df.head(3)

## Subsampling

In [ ]:
# Create is_ig as a True/False masking array
best = techniques_df['best'].values
is_ig = best == 'ig'
is_ig[:20]

In [ ]:
# Use masking array to filter names of IG images
ig_img_names = techniques_df[is_ig].index.values
ig_img_names[:5]

In [ ]:
# Downsample IG images so proportion relative to the whole dataset is near ~60%
from sklearn.model_selection import train_test_split
ig_img_split = train_test_split(ig_img_names, train_size=0.52,
                                         random_state=42, shuffle=True, stratify=None)
selected_ig_img_names = ig_img_split[0]

In [ ]:
# As planned, only ~77 images were selected
len(selected_ig_img_names)

### Creating new subsampled datasets

We'll create a new dataset with the subsampled IG images and the rest of images associated with other techniques. This will be useful for later usage.

In [ ]:
# We already kwow the names of sumsampled IG images
selected_ig_img_names[:5]

In [ ]:
# Then we get the names of images NOT explained with IG
is_not_ig = is_ig == False
not_ig_img_names = techniques_df[is_not_ig].index.values
not_ig_img_names[:5]

In [ ]:
# We join the two arrays of names...
subsampled_img_names = list(selected_ig_img_names) + list(not_ig_img_names)
len(subsampled_img_names)

In [ ]:
# ...and use it to create new datasets (for features and techniques)
sub_feats_df = feats_df.loc[subsampled_img_names]
sub_techniques_df = techniques_df.loc[subsampled_img_names]

In [ ]:
sub_feats_df.head(3)

In [ ]:
sub_techniques_df.head(3)

As we can see, the number of images in the dataset has changed, changing the propotions of techniques

In [ ]:
np.unique(sub_techniques_df['best'].values, return_counts=True)

### Saving new datasets

In [ ]:
# New file names
SUB_FEATS_FILE_PATH = os.path.join('..', 'features', 'sub_incv1_feats.csv')
SUB_BEST_TECHNIQUES = os.path.join('..', 'results', 'sub_votes_summary.csv')

In [ ]:
# Saving datasets
sub_feats_df.to_csv(SUB_FEATS_FILE_PATH)
sub_techniques_df.to_csv(SUB_BEST_TECHNIQUES)